# Tarea grupal: "EnviRápido", centro de distribución de paquetes


**Grupos de 5 · Lenguaje: Python

Este cuaderno contiene el **enunciado**, los **mínimos necesarios**, los **puntos de libertad**, los **entregables** y la **rúbrica**. Al final hay un espacio de trabajo con una celda de código por cada mínimo.



---
# 1. El problema
Una empresa de mensajería, **EnviRápido**, tiene un centro de distribución que hoy funciona con cuadernos y hojas de papel. Su dueño sufre estos problemas:

- Cada día llegan cientos de paquetes y se atienden **en el orden en que llegaron**, pero a veces los empleados se saltan paquetes sin darse cuenta.
- Los clientes llaman preguntando *"¿dónde está mi paquete?"* y el empleado tarda minutos en encontrarlo revisando hoja por hoja.
- A veces un empleado marca un paquete como "entregado" por error y no hay forma de corregirlo ordenadamente.
- La empresa reparte a **zonas/sucursales** (cientos de ellas, con un código numérico) y necesita saber rápidamente a cuál corresponde cada código.
- Hay paquetes que llegan **sin código de seguimiento**, solo con el nombre del cliente, y también hay que poder ubicarlos.

**Tu grupo es una empresa de software** que debe ofrecerle al dueño un prototipo que resuelva estos problemas. Ustedes deciden qué datos tiene un paquete, cómo se organiza el sistema y qué estructuras usar. El dueño no sabe de programación: tendrán que **convencerlo** de que su solución es buena y de por qué.

---
# 2. Mínimos necesarios (para aprobar)

1. **Modelado propio:** definir qué es un paquete, un cliente y una zona (clases o estructuras), y explicar por escrito qué decidieron y por qué.
2. **Recepción y atención en orden de llegada**, con posibilidad de ver quién sigue.
3. **Consulta rápida por código de seguimiento**, usando **una tabla hash implementada por ustedes** (sin usar `dict` ni `set` de Python para la estructura principal), con **manejo de colisiones**.
4. **Corrección de errores** (deshacer acciones, al menos las últimas) usando una estructura adecuada.
5. **Búsqueda de zona por código** sobre una lista ordenada, usando búsqueda binaria.
6. **Búsqueda de paquetes por nombre de cliente** (sin código), usando búsqueda secuencial.
7. **Pruebas:** un programa de ejemplo con al menos 20 paquetes que demuestre cada funcionalidad, incluyendo casos límite (cola vacía, código inexistente, colisión, deshacer sin historial).
8. **Medición:** comparar cuántas comparaciones hace cada búsqueda (secuencial vs. binaria vs. hash) con 100, 1000 y 10000 datos, y mostrar los resultados.

---
# 3. Puntos de libertad (donde tienen que pensar)
Estas decisiones **no las da el enunciado**; deben tomarlas y justificarlas:

- ¿Qué pasa con los paquetes urgentes o con prioridad? ¿Rompen la regla del orden de llegada?
- ¿Qué se hace si dos paquetes tienen el mismo nombre de cliente?
- ¿Cuál es el tamaño de la tabla hash y qué función hash usan? ¿Qué hacen cuando se llena?
- ¿Qué acciones se pueden deshacer y cuáles no?
- ¿La lista de zonas puede cambiar? Si cambia, ¿cómo se mantiene ordenada?
- ¿Cómo reportan al usuario que algo no se encontró?

**Extras opcionales (para nota adicional):** reasignar paquetes entre zonas, estadísticas del día, interfaz de menú por consola, carga de datos desde archivo.

---
# 4. Entregables y "venta"

| Entregable | Detalle |
|---|---|
| Código | Este cuaderno `.ipynb` (o archivos `.py`), comentado y ordenado |
| **Presentación de venta (10 min )** | Demostración en vivo + argumento de por qué su solución es la mejor |

**Reglas de la "venta":** los 5 integrantes deben hablar. El público (el resto de la clase y el docente) actúa como el dueño de la empresa y puede hacer preguntas como *"¿qué pasa si hay 1 millón de paquetes?"* o *"¿por qué no usaron una lista para esto?"*.



---
# ESPACIO DE TRABAJO DEL GRUPO

### Equipo y roles
Completen esta tabla (haciendo doble clic en la celda). Los roles deben **rotar** al preparar la defensa.

| Integrante | Rol principal | Rol de la defensa |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |

### Registro de decisiones de diseño
Escriban aquí, en pocas líneas, qué decidieron para cada punto de libertad y **por qué** (esto alimenta el informe).

- **Paquetes urgentes:**
Se atienden con prioridad adelantándolos en la cola para salir más rápido sin alterar el flujo general.
- **Nombres de cliente repetidos:**
La búsqueda devuelve una lista con todos los paquetes que coinciden con ese nombre para que el empleado elija el correcto.
- **Tamaño y función hash / tabla llena:**
Usamos una capacidad fija y una función hash basada en la suma ASCII módulo la capacidad. Las colisiones se resuelven con listas internas (encadenamiento separado)
- **Qué se puede deshacer y qué no:**
Se pueden revertir cambios de estado erróneos (como marcar un paquete como entregado por error) usando una pila, pero no acciones irreversibles.
- **Zonas (¿cambian? ¿cómo se mantienen ordenadas?):**
Si la lista de zonas cambia o se agrega una nueva, se vuelve a ordenar para mantener la eficacia de la búsqueda binaria.
- **Cómo se informa que algo no se encontró:**
Se retorna None o una lista vacía para indicar claramente que el registro no existe.

### Preparación: módulos
Importen aquí lo que necesiten (por ejemplo `randint` para generar datos de prueba). **Recuerden:** no usar `dict` ni `set` para la tabla hash principal.

In [ ]:
from random import randint  # agreguen aquí los módulos que necesiten
from typing import Optional

### Mínimo 1 · Modelado: paquete, cliente y zona
Definan las clases o estructuras. Antes de programar, escriban en papel los **sustantivos** y los **verbos** del problema.

In [1]:
# Tu código aquí: clases Paquete, Cliente, Zona
class Cliente:
    def __init__(self, nombre):
        self.nombre = nombre

    def __str__(self):
        return f"Cliente('{self.nombre}')"

class Zona:
    def __init__(self, codigo, nombre):
        self.codigo = codigo
        self.nombre = nombre

    def __str__(self):
        return f"Zona({self.codigo}, '{self.nombre}')"

class Paquete:
    def __init__(self, codigo_seguimiento, cliente, estado="En recepción"):
        self.codigo_seguimiento = codigo_seguimiento
        self.cliente = cliente # Guardamos el objeto Cliente
        self.estado = estado


### Mínimo 2 · Recepción y atención en orden de llegada
Necesitan: recibir un paquete, atender el siguiente y **ver quién sigue** sin sacarlo.

In [2]:
# Tu código aquí: estructura para el orden de llegada
class Cola:

    def __init__(self):
        self.queue = []

    def is_empty(self):
        return len(self.queue) == 0

    def enqueue(self, item):
        self.queue.append(item)

    def dequeue(self):
        return None if self.is_empty() else self.queue.pop(0)

    def peek(self):
        return None if self.is_empty() else self.queue[0]

    def size(self):
        return len(self.queue)

    def clear(self):
        self.queue.clear()

    def __len__(self):
        return len(self.queue)

    def __repr__(self):
        return f"Cola({self.queue})"

### Mínimo 3 · Consulta por código de seguimiento (tabla hash propia)
Implementen la tabla hash completa: función hash, insertar, buscar y manejo de colisiones. **Sin `dict` ni `set`.**

In [3]:
# Tu código aquí: tabla hash
class TablaHashPaquetes:
    def __init__(self, capacidad=10):
        self.capacidad = capacidad
        self.buckets = [[] for _ in range(self.capacidad)]
        self.tamanio = 0
        self.comparaciones = 0

    def _funcion_hash(self, codigo):
        #calcula el indice sumando el codigo ascii de los caracteres
        if isinstance(codigo, str):
            suma = sum(ord(c) for c in codigo)
            return suma % self.capacidad
        return int(codigo) % self.capacidad

    def insertar(self, paquete):
        #inserta o actualiza un paquete manejando colisiones por encadenamiento
        indice = self._funcion_hash(paquete.codigo_seguimiento)
        bucket = self.buckets[indice]
        for i, p in enumerate(bucket):
            if p.codigo_seguimiento == paquete.codigo_seguimiento:
                bucket[i] = paquete
                return
        bucket.append(paquete)
        self.tamanio += 1

    def buscar(self, codigo):
        #busca un paquete por su codigo y cuenta las comparaciones
        self.comparaciones = 0
        indice = self._funcion_hash(codigo)
        bucket = self.buckets[indice]
        for p in bucket:
            self.comparaciones += 1
            if p.codigo_seguimiento == codigo:
                return p, self.comparaciones
        return None, self.comparaciones

### Mínimo 4 · Deshacer acciones
Decidan qué acciones se registran y **qué información hay que guardar** para poder revertirlas.

In [4]:
# Tu código aquí: historial y deshacer
class HistorialAcciones:
    def __init__(self):
        self.pila = []

    def registrar_accion(self, tipo, paquete, estado_anterior):
        # guarda la accion y el estado previo en la pila
        accion = {
            "tipo": tipo,
            "paquete": paquete,
            "estado_anterior": estado_anterior
        }
        self.pila.append(accion)

    def deshacer(self):
        #deshacer usando pila
        if not self.pila: #si la pila esta vacía
            return "no hay acciones para deshacer"
        ultima_accion = self.pila.pop()
        paquete = ultima_accion["paquete"]
        estado_previo = ultima_accion["estado_anterior"]
        paquete.estado = estado_previo
        return f"accion deshecha"

### Mínimo 5 · Búsqueda de zona por código (búsqueda binaria)
La lista de zonas debe estar **ordenada** antes de buscar.

In [9]:
# Tu código aquí: lista de zonas ordenada y búsqueda binaria
from typing import Optional

class Busquedas:

    @staticmethod
    def ordenar_zonas(zonas: list[Zona]) -> list[Zona]:
        # ordena la lista de zonas por su código de forma ascendente
        return sorted(zonas, key=lambda x: x.codigo)

    @staticmethod
    def busqueda_binaria_zona(codigo_buscado: int, zonas_ordenadas: list[Zona]) -> tuple[Optional[Zona], int]:
        # realiza búsqueda binaria sobre las zonas ordenadas
        if not zonas_ordenadas:
            return None, 0

        inicio = 0
        fin = len(zonas_ordenadas) - 1
        comparaciones = 0

        while inicio <= fin:
            medio = (inicio + fin) // 2
            comparaciones += 1

            if zonas_ordenadas[medio].codigo == codigo_buscado:
                return zonas_ordenadas[medio], comparaciones
            elif codigo_buscado < zonas_ordenadas[medio].codigo:
                fin = medio - 1
            else:
                inicio = medio + 1

        return None, comparaciones

### Mínimo 6 · Búsqueda de paquetes por nombre de cliente (búsqueda secuencial)
Piensen qué devuelve su búsqueda si hay varios paquetes con el mismo nombre.

In [10]:
# Tu código aquí: búsqueda secuencial por nombre
def busquedaSecuencial(nombre_cliente: str, paquetes: list[Paquete]) -> tuple[list[Paquete], int]:
    # busca paquetes recorriendo la lista uno por uno
    paquetes_encontrados: list[Paquete] = []
    comparaciones = 0

    for p in paquetes:
        comparaciones += 1
        if p.cliente.nombre.strip().lower() == nombre_cliente.strip().lower():
            paquetes_encontrados.append(p)

    return paquetes_encontrados, comparaciones

### Mínimo 7 · Pruebas con al menos 20 paquetes
Demuestren **cada** funcionalidad. Incluyan casos límite: cola vacía, código inexistente, colisión y deshacer sin historial.

In [11]:
# Tu código aquí: programa de pruebas

print("1. DEMOSTRACIÓN CON 20 PAQUETES Y COLA DE RECEPCIÓN ===")
cola_recepcion = Cola()

# Usamos los clientes de prueba y generamos 20 paquetes
clientes_base = [
    Cliente("Ana Gómez"), Cliente("Carlos Ruiz"), Cliente("Lucía Pérez"),
    Cliente("Juan Pérez"), Cliente("María López"), Cliente("Pedro Ruiz")
]

lista_20_paquetes = []
for i in range(1, 21):
    cliente_actual = clientes_base[i % len(clientes_base)] # Se repiten para probar búsquedas múltiples
    codigo = f"PKG-{i:03d}"
    paq = Paquete(codigo, cliente_actual)
    lista_20_paquetes.append(paq)
    cola_recepcion.enqueue(paq)

print(f"Se recibieron {cola_recepcion.size()} paquetes en orden.")
print(f"Viendo quién sigue (peek): {cola_recepcion.peek().codigo_seguimiento}")
atendido = cola_recepcion.dequeue()
print(f"Atendiendo a: {atendido.codigo_seguimiento}. Quedan en cola: {cola_recepcion.size()}")

# Caso límite: extraer de cola vacía
cola_vacia = Cola()
print(f"caso limite - sacar de cola vacía: {cola_vacia.dequeue()}")


print("\n=== 2. PRUEBAS ESPECÍFICAS Y CASOS LÍMITE (Mínimos 3, 4, 5 y 6) ===")

#pruebas para minimo 3
tabla_prueba = TablaHashPaquetes(capacidad=3)

cli1 = clientes_base[0] # Ana Gómez
cli2 = clientes_base[1] # Carlos Ruiz
cli3 = clientes_base[2] # Lucía Pérez

paq1 = Paquete("A1", cli1, "En recepción")
paq2 = Paquete("D4", cli2, "En recepción") # si generan el mismo indice hash, habra colision
paq3 = Paquete("G7", cli3, "En recepción")

tabla_prueba.insertar(paq1)
tabla_prueba.insertar(paq2)
tabla_prueba.insertar(paq3)

#caso normal: buscar codigo existente
resultado, comps = tabla_prueba.buscar("D4")
print(f"busqueda codigo existente ('D4'): {resultado} | comparaciones: {comps}")

#caso limite: codigo inexistente en la tabla hash
resultado_inexistente, comps_inex = tabla_prueba.buscar("Z99")
print(f"caso limite - codigo inexistente ('Z99'): {resultado_inexistente} | comparaciones: {comps_inex}")


#pruebas minimo 4
historial_prueba = HistorialAcciones()

#caso limite: intentar deshacer sin historial previo
print(f"caso limite - deshacer sin historial: {historial_prueba.deshacer()}")

#registrar cambio de estado y deshacer
estado_anterior_paq1 = paq1.estado
paq1.estado = "Entregado" # simulamos error humano
historial_prueba.registrar_accion("cambiar_estado", paq1, estado_anterior_paq1)

print(f"estado actual tras error: {paq1.estado}")
print(historial_prueba.deshacer())
print(f"estado despues de aplicar deshacer: {paq1.estado}")

#Pruebas para busquedas binarias
zonas = [
        Zona(105, "Norte"),
        Zona(101, "Centro"),
        Zona(103, "Sur"),
        Zona(110, "Este"),
    ]

zona, comparaciones = Busquedas.busqueda_binaria_zona(110, Busquedas.ordenar_zonas(zonas))

print(f"Búsqueda 110 → {zona} | comparaciones: {comparaciones}")


#Pruebas para busquedas secuenciales
# Usamos la lista de 20 paquetes generada al principio para que la búsqueda sea más robusta
encontrados, comparaciones = busquedaSecuencial("María López", lista_20_paquetes)
print(f"Búsqueda Secuencial ('María López'): {len(encontrados)} paquetes encontrados | Comparaciones: {comparaciones}")

1. DEMOSTRACIÓN CON 20 PAQUETES Y COLA DE RECEPCIÓN ===
Se recibieron 20 paquetes en orden.
Viendo quién sigue (peek): PKG-001
Atendiendo a: PKG-001. Quedan en cola: 19
caso limite - sacar de cola vacía: None

=== 2. PRUEBAS ESPECÍFICAS Y CASOS LÍMITE (Mínimos 3, 4, 5 y 6) ===
busqueda codigo existente ('D4'): <__main__.Paquete object at 0x7a5d20995ac0> | comparaciones: 2
caso limite - codigo inexistente ('Z99'): None | comparaciones: 3
caso limite - deshacer sin historial: no hay acciones para deshacer
estado actual tras error: Entregado
accion deshecha
estado despues de aplicar deshacer: En recepción
Búsqueda 110 → Zona(110, 'Este') | comparaciones: 3
Búsqueda Secuencial ('María López'): 3 paquetes encontrados | Comparaciones: 20


### Mínimo 8 · Medición de comparaciones
Midan cuántas comparaciones hace cada búsqueda (secuencial, binaria y hash) con **100, 1000 y 10000** datos y muestren una tabla con los resultados. Después escriban en la celda siguiente qué observan.

In [18]:
# Tu código aquí: mediciones

from random import randint

print("MEDICIONES DE RENDIMIENTO ")

for n in [100, 1000, 10000]:
    # 1. Preparar datos
    paquetes_med = [Paquete(f"COD{i}", Cliente(f"Cliente{i}")) for i in range(n)]
    zonas_med = [Zona(i, f"Zona{i}") for i in range(n)]

    tabla_med = TablaHashPaquetes(capacidad = n // 5)
    for p in paquetes_med:
        tabla_med.insertar(p)

    # Índice aleatorio para probar
    idx_random = randint(0, n - 1)

    # --- Búsqueda Secuencial (Nombre del cliente) ---
    # Mejor: primer elemento. Peor: último elemento.
    _, sec_mejor = busquedaSecuencial("Cliente0", paquetes_med)
    _, sec_peor = busquedaSecuencial(f"Cliente{n-1}", paquetes_med)
    _, sec_rand = busquedaSecuencial(f"Cliente{idx_random}", paquetes_med)
    _, sec_nada = busquedaSecuencial("Nadie", paquetes_med)

    # --- Búsqueda Binaria (Código de zona) ---
    # Mejor: elemento en la mitad exacta. Peor: elemento en el extremo (índice 0).
    centro = (n - 1) // 2
    _, bin_mejor = Busquedas.busqueda_binaria_zona(zonas_med[centro].codigo, zonas_med)
    _, bin_peor = Busquedas.busqueda_binaria_zona(zonas_med[0].codigo, zonas_med)
    _, bin_rand = Busquedas.busqueda_binaria_zona(zonas_med[idx_random].codigo, zonas_med)
    _, bin_nada = Busquedas.busqueda_binaria_zona(-1, zonas_med)

    # --- Búsqueda Hash (Código de seguimiento) ---
    # Hash tiende a ser O(1), pero medimos las variaciones reales por encadenamiento
    _, hash_mejor = tabla_med.buscar("COD0")
    _, hash_peor = tabla_med.buscar(f"COD{n-1}")
    _, hash_rand = tabla_med.buscar(f"COD{idx_random}")
    _, hash_nada = tabla_med.buscar("DOC500") # Forzamos la colisión ASCII

    # 2. Imprimir tabla de resultados por cada volumen de datos
    print(f"\nResultados para N = {n}")
    print(f"{'Escenario':<15} | {'Secuencial':<12} | {'Binaria':<10} | {'Hash':<10}")
    print("-" * 55)
    print(f"{'Mejor caso':<15} | {sec_mejor:<12} | {bin_mejor:<10} | {hash_mejor:<10}")
    print(f"{'Peor (Existe)':<15} | {sec_peor:<12} | {bin_peor:<10} | {hash_peor:<10}")
    print(f"{'Aleatorio':<15} | {sec_rand:<12} | {bin_rand:<10} | {hash_rand:<10}")
    print(f"{'No encontrado':<15} | {sec_nada:<12} | {bin_nada:<10} | {hash_nada:<10}")

MEDICIONES DE RENDIMIENTO 

Resultados para N = 100
Escenario       | Secuencial   | Binaria    | Hash      
-------------------------------------------------------
Mejor caso      | 100          | 1          | 1         
Peor (Existe)   | 100          | 6          | 2         
Aleatorio       | 100          | 7          | 3         
No encontrado   | 100          | 6          | 7         

Resultados para N = 1000
Escenario       | Secuencial   | Binaria    | Hash      
-------------------------------------------------------
Mejor caso      | 1000         | 1          | 1         
Peor (Existe)   | 1000         | 9          | 1         
Aleatorio       | 1000         | 10         | 28        
No encontrado   | 1000         | 9          | 15        

Resultados para N = 10000
Escenario       | Secuencial   | Binaria    | Hash      
-------------------------------------------------------
Mejor caso      | 10000        | 1          | 1         
Peor (Existe)   | 10000        | 13        

### Análisis de las mediciones
¿Qué pasa con las comparaciones al multiplicar los datos por 10? ¿Cuál búsqueda crece más rápido? ¿Su tabla hash se comportó como esperaban?


En la búsqueda secuencial las comparaciones aumentan 10 veces de forma proporcional, mientras que en la binaria y la tabla hash apenas incrementan al mínimo. La búsqueda secuencial es la que crece más rápido debido a su comportamiento lineal, el cual depende directamente del tamaño de los datos. La tabla hash demostró una eficiencia excelente con tiempos de búsqueda casi constantes y un buen manejo de colisiones.

### Lista de verificación antes de entregar
- [ ] Los 8 mínimos funcionan y están demostrados
- [ ] La tabla hash no usa `dict` ni `set` y maneja colisiones
- [ ] El código está comentado y ordenado
- [ ] Los 5 integrantes pueden explicar **cualquier** parte, no solo la suya
